# MongoDB Atlas Vector Search with Voyage AI: Embeddings, Automated Embedding, and Reranking

This notebook builds semantic product search on MongoDB Atlas using Voyage AI models.

| Section | What it shows |
|---|---|
| 1. Environment setup | Loading credentials and configuration from a `.env` file |
| 2. Dataset preparation | A small e-commerce catalog, with each product combined into one text string for embedding |
| 3. Approach 1 — Bring Your Own Embeddings (BYOE) | Generating vectors with the Voyage AI embedding API, storing them in Atlas, and querying with `$vectorSearch` + `queryVector` |
| 4. Approach 2 — Automated Embedding | Letting Atlas generate and maintain vectors through an `autoEmbed` index and querying with plain text |
| 5. Reranking — Method 1 | Reranking inside the aggregation pipeline with the native `$rerank` stage |
| 6. Reranking — Method 2 | Reranking in application code by calling the `/rerank` endpoint directly |
| 7. Comparison and cleanup | Comparing rankings side by side and removing demo resources |

### Prerequisites

- A MongoDB Atlas cluster with Vector Search. On dedicated clusters (`M10+`), Automated Embedding requires **storage auto-scaling** to be enabled.
- For the native `$rerank` stage: MongoDB **9.0 or later** and **Native Reranking** turned on in **Project Settings**. The `rerank-3` and `rerank-3-lite` models also require a dedicated (`M10+`) cluster.
- A **model API key** created in Atlas. It authenticates calls to the Atlas Embedding and Reranking API (`https://ai.mongodb.com/v1`).
- The IP address of this machine added to the Atlas IP access list.

> **Naming note:** the aggregation stage is called `$rerank` (not `$reranker`).

## 1. Environment Setup and Configuration

All credentials and settings are read from a `.env` file with `python-dotenv`, so no secret is written in the notebook. Copy `.env.example` to `.env` (next to this notebook) and fill in your values:

```dotenv
# MongoDB Atlas
MONGODB_URI="mongodb+srv://<user>:<password>@<cluster-host>/?retryWrites=true&w=majority"
MONGODB_DB_NAME="ecommerce_demo"

# Atlas Embedding and Reranking API (Voyage AI models)
VOYAGE_API_KEY="<atlas-model-api-key>"
VOYAGE_EMBEDDING_MODEL="voyage-4"   # Bring-Your-Own-Embeddings model
VOYAGE_RERANK_MODEL="rerank-2.5"   # reranker model (rerank-3 / rerank-3-lite need an M10+ cluster for $rerank)

# Optional
AUTOEMBED_MODEL="voyage-4"          # model used by the autoEmbed index (defaults to VOYAGE_EMBEDDING_MODEL)
EMBEDDING_DIMENSIONS="1024"         # 256 | 512 | 1024 | 2048
```

> `.env` is listed in `.gitignore` so credentials are never committed. Only `.env.example` (placeholders) belongs in version control.

In [ ]:
# Install the required packages into the active kernel environment.
# pymongo: MongoDB driver | python-dotenv: reads the .env file | requests: HTTP client for the Embedding and Reranking API
%pip install --quiet pymongo python-dotenv requests

In [ ]:
# Standard library: `os` reads environment variables, `time` is used for retry backoff and index polling.
import os
import time
from typing import Any, Optional

# Third-party: `requests` calls the Voyage AI REST API; `pymongo` talks to Atlas (queries, inserts, search indexes).
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.collection import Collection
from pymongo.errors import ConnectionFailure, OperationFailure, PyMongoError
from pymongo.operations import SearchIndexModel

# Load the .env file. Variables already set in the environment take precedence.
# load_dotenv() returns False when it cannot find a .env file.
if not load_dotenv():
    print("⚠️  No .env file found. Falling back to existing environment variables.")

# Fail fast: stop here with a clear message if a required setting is missing, instead of
# failing later with a less obvious error (for example, a 401 from the API or a DNS error).
REQUIRED_VARS = ["MONGODB_URI", "VOYAGE_API_KEY", "VOYAGE_EMBEDDING_MODEL", "VOYAGE_RERANK_MODEL"]
missing_vars = [name for name in REQUIRED_VARS if not os.getenv(name)]
if missing_vars:
    raise EnvironmentError(f"Missing required environment variables: {', '.join(missing_vars)}")

# Required settings
MONGODB_URI = os.environ["MONGODB_URI"]
VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
VOYAGE_EMBEDDING_MODEL = os.environ["VOYAGE_EMBEDDING_MODEL"]
VOYAGE_RERANK_MODEL = os.environ["VOYAGE_RERANK_MODEL"]

# Optional settings with defaults
AUTOEMBED_MODEL = os.getenv("AUTOEMBED_MODEL", VOYAGE_EMBEDDING_MODEL)
VOYAGE_API_BASE_URL = os.getenv("VOYAGE_API_BASE_URL", "https://ai.mongodb.com/v1").rstrip("/")
DB_NAME = os.getenv("MONGODB_DB_NAME", "ecommerce_demo")
# The vector size must be identical for stored vectors, query vectors, and the index definition.
# Voyage 4 models support 256, 512, 1024 (default), and 2048 dimensions: smaller = less storage, larger = more detail.
# Docs (output_dimension per model): https://www.mongodb.com/docs/api/doc/atlas-embedding-and-reranking-api/operation/operation-createembedding
try:
    EMBEDDING_DIMENSIONS = int(os.getenv("EMBEDDING_DIMENSIONS", "1024"))
except ValueError as exc:
    raise ValueError("EMBEDDING_DIMENSIONS must be an integer (256, 512, 1024, or 2048).") from exc
if EMBEDDING_DIMENSIONS not in {256, 512, 1024, 2048}:
    raise ValueError(f"Unsupported EMBEDDING_DIMENSIONS={EMBEDDING_DIMENSIONS}. Use 256, 512, 1024, or 2048.")

# Collection and index names (one collection per approach keeps the demos independent)
BYOE_COLLECTION_NAME = "products_byoe"
AUTO_COLLECTION_NAME = "products_autoembed"
BYOE_INDEX_NAME = "byoe_vector_index"
AUTO_INDEX_NAME = "autoembed_vector_index"

# Print non-secret configuration only
# The connection string and API key are intentionally never printed.
print("✅ Configuration loaded")
print(f"   Database:              {DB_NAME}")
print(f"   BYOE embedding model:  {VOYAGE_EMBEDDING_MODEL} ({EMBEDDING_DIMENSIONS} dims)")
print(f"   autoEmbed model:       {AUTOEMBED_MODEL}")
print(f"   Reranker model:        {VOYAGE_RERANK_MODEL}")
print(f"   API base URL:          {VOYAGE_API_BASE_URL}")

### 1.1 Connect to MongoDB Atlas

The client pings the cluster right away so connection problems (wrong URI, IP not on the access list, bad credentials) surface here instead of later in the notebook.

In [ ]:
def get_mongo_client(uri: str) -> MongoClient:
    """Create a MongoClient and verify connectivity with a ping."""
    # Docs (MongoClient options): https://www.mongodb.com/docs/languages/python/pymongo-driver/current/connect/mongoclient/
    # Docs (connection string format): https://www.mongodb.com/docs/manual/reference/connection-string/
    # Docs (Atlas IP access list): https://www.mongodb.com/docs/atlas/security/ip-access-list/
    # appname tags this connection so it can be identified in Atlas logs and metrics.
    # serverSelectionTimeoutMS caps how long the driver waits for a reachable server (10 seconds).
    client = MongoClient(uri, appname="atlas-voyage-autoembed-rerank-demo", serverSelectionTimeoutMS=10_000)
    try:
        # MongoClient connects lazily; the ping forces a round trip that validates the URI,
        # the credentials, and network access (IP access list) right now.
        client.admin.command("ping")
    except (ConnectionFailure, OperationFailure) as exc:
        client.close()
        raise ConnectionError(
            "Could not connect to MongoDB Atlas. Check MONGODB_URI, credentials, and the IP access list."
        ) from exc
    return client


# One client for the whole notebook: MongoClient keeps a connection pool and is meant to be reused.
mongo_client = get_mongo_client(MONGODB_URI)
db = mongo_client[DB_NAME]
# Collection handles are lightweight references; the collections are created on the first insert.
byoe_collection = db[BYOE_COLLECTION_NAME]
auto_collection = db[AUTO_COLLECTION_NAME]

# buildInfo returns the server version, which tells whether $rerank is supported (requires MongoDB 9.0+).
# Docs (buildInfo): https://www.mongodb.com/docs/manual/reference/command/buildInfo/
server_version = db.command("buildInfo").get("version", "unknown")
print(f"✅ Connected to MongoDB Atlas (server version {server_version})")

### 1.2 Atlas Embedding and Reranking API helpers

Both the BYOE approach and reranking Method 2 call the Atlas Embedding and Reranking API (`POST /embeddings` and `POST /rerank`). The helper below:

- sends the model API key as a `Bearer` token in the `Authorization` header (the key is never printed);
- retries with exponential backoff on network errors and on retryable status codes (`429`, `500`, `502`, `503`, `504`);
- raises a clear error with the API `detail` message for non-retryable errors such as `400`, `401`, or `403`.

In [ ]:
# HTTP status codes that usually mean a temporary problem and are worth retrying:
# 429 = rate limit exceeded (per-key tokens or requests per minute); 5xx = transient server/gateway errors.
# Docs (rate limits): http://dochub.mongodb.org/core/voyage-rate-limits
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}


def call_voyage_api(endpoint: str, payload: dict, max_retries: int = 4, timeout_s: int = 30) -> dict:
    """POST a JSON payload to the Atlas Embedding and Reranking API with retries."""
    # Docs (API overview and authentication): https://www.mongodb.com/docs/api/doc/atlas-embedding-and-reranking-api/
    # Docs (create and manage model API keys): http://dochub.mongodb.org/core/voyage-api-keys
    # Full endpoint URL, for example https://ai.mongodb.com/v1/embeddings or .../v1/rerank
    url = f"{VOYAGE_API_BASE_URL}/{endpoint.lstrip('/')}"
    # The model API key travels as a Bearer token in the Authorization header.
    headers = {"Authorization": f"Bearer {VOYAGE_API_KEY}", "Content-Type": "application/json"}

    for attempt in range(1, max_retries + 1):
        # Network-level failures (DNS, connection reset, timeout) raise before any HTTP response exists.
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=timeout_s)
        except requests.RequestException as exc:
            if attempt == max_retries:
                raise RuntimeError(f"Network error calling /{endpoint} after {attempt} attempts: {exc}") from exc
            # Exponential backoff: wait 2, 4, 8, ... seconds so a struggling service has time to recover.
            wait_s = 2 ** attempt
            print(f"⚠️  Network error on /{endpoint} (attempt {attempt}); retrying in {wait_s}s")
            time.sleep(wait_s)
            continue

        # 2xx: return the parsed JSON body.
        if response.ok:
            return response.json()

        # Temporary error: wait and try again, unless this was the last allowed attempt.
        if response.status_code in RETRYABLE_STATUS_CODES and attempt < max_retries:
            wait_s = 2 ** attempt
            print(f"⚠️  /{endpoint} returned {response.status_code} (attempt {attempt}); retrying in {wait_s}s")
            time.sleep(wait_s)
            continue

        # Permanent error (400 bad request, 401 invalid key, 403 IP not allowed, ...): retrying will not help,
        # so raise with the API's `detail` message, which explains the cause.
        try:
            detail = response.json().get("detail", response.text)
        except ValueError:
            detail = response.text
        raise RuntimeError(f"/{endpoint} failed with HTTP {response.status_code}: {detail}")

    raise RuntimeError(f"/{endpoint} failed after {max_retries} attempts")


def embed_texts(texts: list[str], input_type: str, model: Optional[str] = None, batch_size: int = 128) -> list[list[float]]:
    """Embed texts with a Voyage AI model. input_type must be 'document' (for storage) or 'query' (for search)."""
    # Docs (POST /embeddings): https://www.mongodb.com/docs/api/doc/atlas-embedding-and-reranking-api/operation/operation-createembedding
    # input_type tells the model what the text is: 'document' for content being stored, 'query' for search text.
    # Voyage prepends a short task prompt for each type, which improves retrieval quality.
    if input_type not in {"document", "query"}:
        raise ValueError("input_type must be 'document' or 'query'")
    if not texts:
        return []

    embeddings: list[list[float]] = []
    # The API accepts up to 1,000 inputs per request (with token limits), so large lists are sent in batches.
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = call_voyage_api(
            "embeddings",
            {
                "input": batch,
                "model": model or VOYAGE_EMBEDDING_MODEL,
                "input_type": input_type,
                # Must equal numDimensions in the BYOE index definition, otherwise queries fail.
                "output_dimension": EMBEDDING_DIMENSIONS,
            },
        )
        # Sort by index so each vector lines up with its input text
        data = sorted(response.get("data", []), key=lambda item: item["index"])
        embeddings.extend(item["embedding"] for item in data)

    # Guard against a partial response so vectors can never be attached to the wrong product.
    if len(embeddings) != len(texts):
        raise RuntimeError(f"Expected {len(texts)} embeddings but received {len(embeddings)}")
    return embeddings


# Quick check that the API key and model work before going further
# A failure here usually means an invalid API key (401), an IP that is not allowed (403),
# or an unknown model name (400), and it is cheaper to find out now than mid-notebook.
probe_vector = embed_texts(["connectivity check"], input_type="query")[0]
if len(probe_vector) != EMBEDDING_DIMENSIONS:
    raise RuntimeError(f"Expected {EMBEDDING_DIMENSIONS} dimensions, got {len(probe_vector)}")
print(f"✅ Embedding API reachable: {VOYAGE_EMBEDDING_MODEL} returned a {len(probe_vector)}-dimension vector")

## 2. Dataset Preparation

The catalog below is small and self-contained. It includes some near-duplicates on purpose (for example, everyday sneakers vs. running shoes, earbuds vs. headphones) so the difference between vector-only ranking and reranked results is easier to see.

Each product's attributes are combined into one string, `embedding_text`. Embedding a single string that carries name, brand, category, price, rating, availability, tags, and description lets one vector capture the whole product, and gives the reranker the same full context.

In [ ]:
# Sample catalog. Structured fields (category, price, in_stock) are kept as separate fields so they can be
# used as vector search pre-filters; the text fields are also combined into `embedding_text` below.
# Similar items (e.g. trail shoes vs. casual sneakers) are included so reranking has visible work to do.
RAW_PRODUCTS: list[dict[str, Any]] = [
    {"product_id": "P001", "name": "TrailBlaze Pro Running Shoes", "brand": "StrideLab", "category": "Footwear", "price": 139.99, "rating": 4.7, "in_stock": True,
     "tags": ["running", "trail", "cushioned", "breathable"],
     "description": "Lightweight trail running shoe with responsive foam cushioning, a breathable mesh upper, and a grippy outsole for long-distance runs on mixed terrain."},
    {"product_id": "P002", "name": "CloudStep Everyday Sneakers", "brand": "UrbanFeet", "category": "Footwear", "price": 89.00, "rating": 4.3, "in_stock": True,
     "tags": ["casual", "walking", "memory foam"],
     "description": "Casual lifestyle sneaker with a memory-foam insole for all-day comfort while walking around the city. Not designed for athletic training."},
    {"product_id": "P003", "name": "Marathon Elite Carbon Racer", "brand": "StrideLab", "category": "Footwear", "price": 249.99, "rating": 4.8, "in_stock": False,
     "tags": ["running", "racing", "carbon plate", "marathon"],
     "description": "Carbon-plated racing shoe built for marathon race day, with ultra-light foam that maximizes energy return over long distances."},
    {"product_id": "P004", "name": "ThermoSip Insulated Travel Mug", "brand": "HearthWare", "category": "Kitchen", "price": 29.95, "rating": 4.6, "in_stock": True,
     "tags": ["coffee", "travel", "insulated", "leak-proof"],
     "description": "Double-wall vacuum-insulated stainless steel mug that keeps coffee hot for 8 hours. The leak-proof lid fits car cup holders for the daily commute."},
    {"product_id": "P005", "name": "BrewMaster Pour-Over Kettle", "brand": "HearthWare", "category": "Kitchen", "price": 64.50, "rating": 4.5, "in_stock": True,
     "tags": ["coffee", "tea", "kettle", "temperature control"],
     "description": "Gooseneck electric kettle with precise temperature control for brewing pour-over coffee and tea at home."},
    {"product_id": "P006", "name": "QuietZone ANC Headphones", "brand": "SonicArc", "category": "Electronics", "price": 199.00, "rating": 4.6, "in_stock": True,
     "tags": ["headphones", "noise cancelling", "wireless", "video calls"],
     "description": "Over-ear wireless headphones with adaptive active noise cancellation, 40-hour battery life, and dual beamforming microphones for clear video calls."},
    {"product_id": "P007", "name": "FocusBuds True Wireless Earbuds", "brand": "SonicArc", "category": "Electronics", "price": 129.00, "rating": 4.4, "in_stock": True,
     "tags": ["earbuds", "noise cancelling", "workout", "travel"],
     "description": "Compact earbuds with hybrid noise cancellation, a transparency mode, and a pocketable charging case for workouts and travel."},
    {"product_id": "P008", "name": "ErgoFlex Standing Desk", "brand": "Workwell", "category": "Home Office", "price": 449.00, "rating": 4.7, "in_stock": True,
     "tags": ["desk", "ergonomic", "height adjustable", "home office"],
     "description": "Electric height-adjustable standing desk with dual motors, memory presets, and a cable tray for a clutter-free home office."},
    {"product_id": "P009", "name": "LumaDesk LED Monitor Light", "brand": "Workwell", "category": "Home Office", "price": 59.99, "rating": 4.5, "in_stock": True,
     "tags": ["lighting", "monitor", "eye strain", "home office"],
     "description": "Screen-mounted light bar with adjustable color temperature that reduces eye strain during late-night work sessions without glare on the monitor."},
    {"product_id": "P010", "name": "Summit 2-Person Backpacking Tent", "brand": "PeakTrail", "category": "Outdoor", "price": 329.00, "rating": 4.6, "in_stock": True,
     "tags": ["camping", "backpacking", "ultralight", "waterproof"],
     "description": "Ultralight freestanding tent weighing 1.3 kg with a waterproof rainfly and two doors, ideal for multi-day backpacking trips."},
    {"product_id": "P011", "name": "HydroTrail Running Vest", "brand": "PeakTrail", "category": "Fitness", "price": 74.99, "rating": 4.5, "in_stock": True,
     "tags": ["running", "hydration", "trail", "ultramarathon"],
     "description": "Lightweight hydration vest with two soft flasks and stretch pockets for gels and a phone during long trail runs and ultramarathons."},
    {"product_id": "P012", "name": "CoreBalance Yoga Mat", "brand": "ZenForm", "category": "Fitness", "price": 48.00, "rating": 4.4, "in_stock": True,
     "tags": ["yoga", "pilates", "non-slip", "natural rubber"],
     "description": "Non-slip 6 mm natural rubber yoga mat with alignment lines and cushioning that protects joints during yoga and pilates."},
]

# Every product must have these fields, otherwise build_embedding_text() would fail or produce incomplete text.
REQUIRED_PRODUCT_FIELDS = {"product_id", "name", "brand", "category", "price", "rating", "in_stock", "tags", "description"}


def build_embedding_text(product: dict[str, Any]) -> str:
    """Combine all relevant product attributes into one string for embedding and reranking."""
    # Labelled "Key: value." segments give the model clear context (e.g. that 139.99 is a price, not a size).
    # Booleans become words because embedding models understand "Out of stock" better than `False`.
    # The same string is used for BYOE embeddings, the autoEmbed index, and the reranker, so all three
    # approaches compare like with like.
    availability = "In stock" if product["in_stock"] else "Out of stock"
    return (
        f"Product: {product['name']}. "
        f"Brand: {product['brand']}. "
        f"Category: {product['category']}. "
        f"Price: ${product['price']:.2f}. "
        f"Rating: {product['rating']}/5. "
        f"Availability: {availability}. "
        f"Tags: {', '.join(product['tags'])}. "
        f"Description: {product['description']}"
    )


def prepare_products(raw_products: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Validate each product and return copies that include the combined `embedding_text` field."""
    # Copies are returned ({**product, ...}) so RAW_PRODUCTS is never modified and the cell can be re-run safely.
    prepared = []
    for product in raw_products:
        missing = REQUIRED_PRODUCT_FIELDS - product.keys()
        if missing:
            raise ValueError(f"Product {product.get('product_id', '<unknown>')} is missing fields: {sorted(missing)}")
        prepared.append({**product, "embedding_text": build_embedding_text(product)})
    return prepared


# Build the final product list used by both approaches and show one example string.
products = prepare_products(RAW_PRODUCTS)
print(f"✅ Prepared {len(products)} products. Example embedding_text:")
print(products[0]["embedding_text"])

### 2.1 Shared helpers

These helpers are used by both approaches:

- `reset_collection` drops and reloads a collection so the notebook can be re-run safely.
- `create_vector_index` drops any index with the same name, creates the new index, and waits until Atlas reports it as `READY` and queryable. Index builds are asynchronous, so querying too early returns empty results.
- `print_results` shows ranked results in a compact table.

In [ ]:
def reset_collection(collection: Collection, documents: list[dict[str, Any]]) -> None:
    """Drop the collection and insert fresh copies of the documents."""
    # Docs (drop): https://www.mongodb.com/docs/manual/reference/method/db.collection.drop/
    # Docs (insertMany): https://www.mongodb.com/docs/manual/reference/method/db.collection.insertMany/
    # Dropping first makes the cell idempotent: re-running never creates duplicate products.
    # Dropping a collection also drops its search indexes. Copies are inserted because
    # insert_many() adds an `_id` to each dict it receives.
    try:
        collection.drop()
        result = collection.insert_many([dict(doc) for doc in documents])
    except PyMongoError as exc:
        raise RuntimeError(f"Failed to load data into '{collection.name}': {exc}") from exc
    print(f"✅ Inserted {len(result.inserted_ids)} documents into '{collection.name}'")


def drop_search_index_if_exists(collection: Collection, index_name: str, timeout_s: int = 180) -> None:
    """Drop a search index by name (if present) and wait until it is gone."""
    # Docs (manage search indexes with PyMongo): https://www.mongodb.com/docs/languages/python/pymongo-driver/current/indexes/atlas-search-index/
    # Search index operations are asynchronous: drop_search_index() returns immediately, so the loop
    # waits until the index is really gone before a new index with the same name can be created.
    if not list(collection.list_search_indexes(index_name)):
        return
    collection.drop_search_index(index_name)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if not list(collection.list_search_indexes(index_name)):
            print(f"🗑️  Dropped existing index '{index_name}'")
            return
        time.sleep(3)
    raise TimeoutError(f"Index '{index_name}' was not dropped within {timeout_s}s")


def wait_for_search_index(collection: Collection, index_name: str, timeout_s: int = 600, poll_s: int = 5) -> dict:
    """Poll until the index is READY and queryable. Raise if it fails or times out."""
    # Docs (index status values and `queryable`): https://www.mongodb.com/docs/manual/reference/method/db.collection.getSearchIndexes/
    # Typical status progression: PENDING -> BUILDING -> READY. `queryable` becomes True once the index
    # can serve queries; querying before that returns empty results rather than an error.
    deadline = time.time() + timeout_s
    last_status = None
    while time.time() < deadline:
        indexes = list(collection.list_search_indexes(index_name))
        # The list is empty for a moment right after creation, until Atlas registers the new index.
        if indexes:
            info = indexes[0]
            status = info.get("status")
            if status == "FAILED":
                raise RuntimeError(f"Index '{index_name}' failed to build: {info.get('message', 'no details provided')}")
            if status == "READY" and info.get("queryable"):
                print(f"✅ Index '{index_name}' is READY and queryable")
                return info
            # Print only status changes to keep the output short while waiting.
            if status != last_status:
                print(f"⏳ Index '{index_name}' status: {status}")
                last_status = status
        time.sleep(poll_s)
    raise TimeoutError(f"Index '{index_name}' was not ready within {timeout_s}s (last status: {last_status})")


def create_vector_index(collection: Collection, index_name: str, definition: dict, timeout_s: int = 600) -> dict:
    """Create (or recreate) a vectorSearch index and wait for it to become queryable."""
    # Docs (vectorSearch index definition): https://www.mongodb.com/docs/vector-search/indexes/vector-search-type/
    drop_search_index_if_exists(collection, index_name)
    # type="vectorSearch" distinguishes a vector index from a full-text Atlas Search index (type="search").
    model = SearchIndexModel(definition=definition, name=index_name, type="vectorSearch")
    try:
        collection.create_search_index(model=model)
    except OperationFailure as exc:
        # Common causes of failure: invalid definition, wrong field type, or an unsupported embedding model.
        raise RuntimeError(f"Failed to create index '{index_name}' on '{collection.name}': {exc}") from exc
    print(f"🛠️  Creating index '{index_name}' on '{collection.name}'")
    return wait_for_search_index(collection, index_name, timeout_s=timeout_s)


def print_results(title: str, results: list[dict[str, Any]], score_fields: tuple[str, ...] = ("score",)) -> None:
    """Print ranked results with the requested score fields."""
    print()
    print(title)
    print("-" * len(title))
    if not results:
        print("  (no results)")
        return
    for rank, doc in enumerate(results, start=1):
        # Show each requested score (e.g. vector_score and rerank_score) when present on the document.
        scores = "  ".join(f"{field}={doc[field]:.4f}" for field in score_fields if field in doc)
        print(f"  {rank:>2}. {doc['name']:<34} {doc['category']:<12} ${doc['price']:>7.2f}  {scores}")


# Fields returned by every search in this notebook (the raw embedding vector is excluded)
# Excluding the vector keeps the results small and readable: one 1024-dimension vector is roughly 8 KB of output.
RESULT_PROJECTION = {"_id": 0, "product_id": 1, "name": 1, "category": 1, "price": 1, "embedding_text": 1}

## 3. Approach 1 — Bring Your Own Embeddings (BYOE)

With BYOE, the application owns the embedding lifecycle:

1. Call the embedding API with `input_type="document"` for each product's `embedding_text`.
2. Store the returned vector in the document (field `embedding`).
3. Create a `vectorSearch` index with a `vector` field (dimensions and similarity must match the model output) plus `filter` fields for pre-filtering.
4. At query time, embed the query with `input_type="query"` and pass it as `queryVector` to `$vectorSearch`.

This gives full control over the model, dimensions, and when vectors are refreshed. The application must also re-embed documents whenever their text changes.

In [ ]:
# Step 1-2: generate document embeddings and store them alongside each product
# input_type="document" optimizes the vectors for being *found*; queries later use input_type="query".
# All 12 products go in a single API call (embed_texts batches automatically for larger lists).
document_vectors = embed_texts([p["embedding_text"] for p in products], input_type="document")
# zip() pairs each product with its vector in the same order; the vector is stored in the `embedding` field.
byoe_documents = [{**product, "embedding": vector} for product, vector in zip(products, document_vectors)]
print(f"✅ Generated {len(document_vectors)} embeddings with {VOYAGE_EMBEDDING_MODEL}")

# Load the documents (text + vector) into the BYOE collection, replacing any previous run.
reset_collection(byoe_collection, byoe_documents)

In [ ]:
# Step 3: create a vectorSearch index on the stored vectors, plus filter fields
# - "vector": the field holding the vectors. numDimensions must equal the embedding size, and
#   "cosine" similarity compares vector direction (the usual choice for Voyage embeddings).
# - "filter": fields that $vectorSearch can use to pre-filter candidates (category, price range, stock).
# Docs (vector and filter field types): https://www.mongodb.com/docs/vector-search/indexes/vector-search-type/
byoe_index_definition = {
    "fields": [
        {"type": "vector", "path": "embedding", "numDimensions": EMBEDDING_DIMENSIONS, "similarity": "cosine"},
        {"type": "filter", "path": "category"},
        {"type": "filter", "path": "price"},
        {"type": "filter", "path": "in_stock"},
    ]
}

# Creates the index and blocks until it is READY (usually under a minute for a small collection).
create_vector_index(byoe_collection, BYOE_INDEX_NAME, byoe_index_definition)

### 3.1 Query with `$vectorSearch` and `queryVector`

`build_vector_search_stage` builds the `$vectorSearch` stage for either approach, so later sections (including reranking) can reuse it:

- `"byoe"` embeds the query in the application and sends `queryVector`.
- `"auto"` sends the raw text in `query.text` and lets Atlas embed it (see Section 4).

`numCandidates` controls how many nearest neighbours the approximate (HNSW) search considers before returning the top `limit`. A value of 10–20× `limit` is a common starting point.

In [ ]:
# Maps the approach name to its collection so the same helper functions work for both approaches.
COLLECTIONS = {"byoe": byoe_collection, "auto": auto_collection}


def build_vector_search_stage(approach: str, query_text: str, limit: int, num_candidates: int,
                              filter_: Optional[dict] = None) -> dict:
    """Build a $vectorSearch stage for the 'byoe' (queryVector) or 'auto' (query.text) approach."""
    # Docs ($vectorSearch fields: queryVector, query.text, limit, numCandidates, filter):
    # https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/
    if not query_text or not query_text.strip():
        raise ValueError("query_text must be a non-empty string")
    if approach == "byoe":
        # BYOE: the application embeds the query (input_type="query") with the same model used for the
        # documents and sends the resulting numbers as `queryVector`.
        stage = {
            "index": BYOE_INDEX_NAME,
            "path": "embedding",
            "queryVector": embed_texts([query_text], input_type="query")[0],
        }
    elif approach == "auto":
        # autoEmbed: send plain text in `query.text`; Atlas embeds it with the index's model.
        # `path` is the text field declared as type "autoEmbed" in the index, not a vector field.
        stage = {
            "index": AUTO_INDEX_NAME,
            "path": "embedding_text",
            "query": {"text": query_text},
        }
    else:
        raise ValueError(f"Unknown approach '{approach}'. Use 'byoe' or 'auto'.")

    # limit = number of results returned. numCandidates = how many nearest neighbours the approximate
    # (HNSW) search explores first; higher means better recall but slower. It can never be below limit.
    stage["limit"] = limit
    stage["numCandidates"] = max(num_candidates, limit)
    # Optional pre-filter (MQL syntax). Filtered fields must be indexed as type "filter".
    # Docs (supported filter operators): https://www.mongodb.com/docs/vector-search/query/aggregation-stages/vector-search-stage/#std-label-vectorSearch-agg-pipeline-filter
    if filter_:
        stage["filter"] = filter_
    return {"$vectorSearch": stage}


def vector_search(approach: str, query_text: str, limit: int = 5, num_candidates: int = 100,
                  filter_: Optional[dict] = None) -> list[dict[str, Any]]:
    """Run a vector-only search and return documents with their vectorSearchScore as `score`."""
    # Docs ($project): https://www.mongodb.com/docs/manual/reference/operator/aggregation/project/
    # Docs ($meta): https://www.mongodb.com/docs/manual/reference/operator/aggregation/meta/
    # $vectorSearch must be the first stage of the pipeline. $project then keeps only readable fields
    # and exposes the similarity score (0-1, higher is more similar) via {"$meta": "vectorSearchScore"}.
    pipeline = [
        build_vector_search_stage(approach, query_text, limit, num_candidates, filter_),
        {"$project": {**RESULT_PROJECTION, "score": {"$meta": "vectorSearchScore"}}},
    ]
    try:
        # Usual causes: wrong index name, index not READY, wrong vector size, or a filter on a non-indexed field.
        return list(COLLECTIONS[approach].aggregate(pipeline))
    except OperationFailure as exc:
        raise RuntimeError(f"$vectorSearch failed for approach '{approach}': {exc}") from exc


# Natural-language queries that share few exact words with the product text, to show semantic matching.
SAMPLE_QUERIES = [
    "comfortable shoes for long-distance trail running",
    "keep my coffee hot during the morning commute",
    "block out noise while working from home on video calls",
]

for query in SAMPLE_QUERIES:
    print_results(f"BYOE | {query}", vector_search("byoe", query, limit=3))

### 3.2 Vector search with pre-filtering

Fields indexed as `filter` can narrow the candidate set *before* the similarity search. Here only in-stock footwear under $150 is considered.

In [ ]:
# Pre-filter: only in-stock Footwear priced at $150 or less is considered for the similarity search.
# This excludes the Marathon Elite Carbon Racer ($249.99, out of stock) even though it is a marathon shoe.
footwear_filter = {"$and": [{"category": "Footwear"}, {"price": {"$lte": 150}}, {"in_stock": True}]}
print_results(
    "BYOE + filter | running shoes (in-stock Footwear <= $150)",
    vector_search("byoe", "running shoes for a marathon", limit=3, filter_=footwear_filter),
)

## 4. Approach 2 — Automated Embedding (`autoEmbed`)

With Automated Embedding, Atlas owns the embedding lifecycle:

- Documents are stored **without** any vector field. Only the text field (`embedding_text`) is needed.
- The index declares that field with `"type": "autoEmbed"`, `"modality": "text"`, and a Voyage AI `model`. Atlas embeds existing documents at index build time and re-embeds documents automatically on insert and update.
- At query time, `$vectorSearch` receives plain text in `query.text`. Atlas embeds the query with the same model as the index.

No embedding code runs in the application. The first build can take longer than a BYOE index because Atlas generates the embeddings during the build.

In [ ]:
# Store the products as-is: no embedding field
# No embedding API call happens here: only text and structured fields are written to MongoDB.
# Docs (Automated Embedding overview): https://www.mongodb.com/docs/vector-search/crud-embeddings/automated-embedding/
reset_collection(auto_collection, products)

# - "autoEmbed": Atlas reads `embedding_text`, generates vectors with the given Voyage model, stores them
#   internally, and keeps them in sync on every insert/update. No numDimensions is needed: the model sets it.
# - "filter": same pre-filter fields as the BYOE index, so both approaches support the same filters.
# Docs (autoEmbed index fields): https://www.mongodb.com/docs/vector-search/indexes/vector-search-type/
# Docs (supported embedding models): https://www.mongodb.com/docs/vector-search/crud-embeddings/automated-embedding/models/
auto_index_definition = {
    "fields": [
        {"type": "autoEmbed", "modality": "text", "path": "embedding_text", "model": AUTOEMBED_MODEL},
        {"type": "filter", "path": "category"},
        {"type": "filter", "path": "price"},
        {"type": "filter", "path": "in_stock"},
    ]
}

# Allow extra time because Atlas generates the embeddings during the initial build
create_vector_index(auto_collection, AUTO_INDEX_NAME, auto_index_definition, timeout_s=900)

In [ ]:
# Query with plain text: Atlas embeds the query using the index's model
# Same queries as BYOE; the only difference in the pipeline is `query.text` instead of `queryVector`.
for query in SAMPLE_QUERIES:
    print_results(f"autoEmbed | {query}", vector_search("auto", query, limit=3))

### 4.1 Automatic embedding of new documents

Inserting a new product into the `autoEmbed` collection is enough to make it searchable. Atlas generates its embedding in the background, so the cell polls for a short time until the new product appears. With BYOE, the application would have to call the embedding API and store the vector itself.

In [ ]:
# A new product that is not in the original catalog.
NEW_PRODUCT = {
    "product_id": "P013", "name": "PressGo Portable Coffee Maker", "brand": "HearthWare", "category": "Kitchen",
    "price": 39.95, "rating": 4.7, "in_stock": True, "tags": ["coffee", "portable", "camping", "travel"],
    "description": "Compact hand-powered coffee press that brews smooth espresso-style coffee anywhere: at the office, on the trail, or while camping.",
}
new_product = prepare_products([NEW_PRODUCT])[0]

# Upsert by product_id so re-running the cell updates the same document instead of adding duplicates.
# Only text is written; Atlas detects the change and generates the embedding in the background.
# Docs (replaceOne with upsert): https://www.mongodb.com/docs/manual/reference/method/db.collection.replaceOne/
# Docs (how Automated Embedding handles inserts and updates): https://www.mongodb.com/docs/vector-search/crud-embeddings/automated-embedding/overview/
try:
    auto_collection.replace_one({"product_id": new_product["product_id"]}, new_product, upsert=True)
except PyMongoError as exc:
    raise RuntimeError(f"Failed to insert the new product: {exc}") from exc
print(f"➕ Inserted '{new_product['name']}' without an embedding")

# Embedding happens asynchronously, so poll the search (every 5 seconds, for up to 2 minutes)
# until the new product appears in the top results.
sync_query = "portable coffee maker for camping trips"
deadline = time.time() + 120
while True:
    sync_results = vector_search("auto", sync_query, limit=3)
    if any(doc["product_id"] == new_product["product_id"] for doc in sync_results):
        print("✅ New product was embedded automatically and is now searchable")
        break
    if time.time() > deadline:
        print("⚠️  New product not visible yet. Embedding generation may still be in progress; re-run this cell shortly.")
        break
    time.sleep(5)

print_results(f"autoEmbed | {sync_query}", sync_results)

## 5. Reranking — Method 1: Native `$rerank` Aggregation Stage

Vector search is fast but scores the query and each document separately. A **reranker** is a cross-encoder model: it reads the query and each candidate document together and assigns a more precise relevance score. The usual pattern is two stages:

1. **Retrieve** a wider candidate set with `$vectorSearch` (for example, the top 10).
2. **Rerank** those candidates and keep the best few.

The `$rerank` stage runs step 2 inside the aggregation pipeline, so no extra API call is made from the application:

```json
{"$rerank": {"query": {"text": "<query>"}, "path": "embedding_text", "numDocsToRerank": 10, "model": "rerank-2.5"}}
```

- `path` is the text the reranker reads. Every input document must have this field.
- `numDocsToRerank` (≤ 1000) caps how many incoming documents are sent to the model.
- The rerank score is read with `{"$meta": "score"}`. The original vector score is copied into `vector_score` *before* `$rerank` so both can be compared.

> **Requirements:** MongoDB 9.0+ and **Native Reranking** enabled in Project Settings. `rerank-3` / `rerank-3-lite` also need an `M10+` dedicated cluster. `$rerank` is not available on self-managed or Atlas Local deployments.

In [ ]:
# RERANK_CANDIDATES: how many vector search results are passed to the reranker (wider net = more chances
# to find the best item, but more reranking cost). TOP_N: how many results are kept after reranking.
# Docs ($rerank stage): https://www.mongodb.com/docs/vector-search/query/aggregation-stages/rerank/
# Docs (enable Native Reranking in Project Settings): https://www.mongodb.com/docs/vector-search/query/native-reranking/managing/
RERANK_CANDIDATES = 10
TOP_N = 3


def vector_search_with_native_rerank(approach: str, query_text: str, top_n: int = TOP_N,
                                     num_docs_to_rerank: int = RERANK_CANDIDATES,
                                     filter_: Optional[dict] = None) -> list[dict[str, Any]]:
    """Retrieve candidates with $vectorSearch, then rerank them in-database with $rerank."""
    if not 1 <= num_docs_to_rerank <= 1000:
        raise ValueError("num_docs_to_rerank must be between 1 and 1000")
    pipeline = [
        # 1. Retrieve: vector search returns RERANK_CANDIDATES documents, ordered by vector similarity.
        build_vector_search_stage(approach, query_text, limit=num_docs_to_rerank,
                                  num_candidates=num_docs_to_rerank * 10, filter_=filter_),
        # 2. Copy the vector score into a regular field now, so it is still available after $rerank
        #    reorders the documents and can be compared with the rerank score.
        #    Docs: https://www.mongodb.com/docs/manual/reference/operator/aggregation/addFields/
        {"$addFields": {"vector_score": {"$meta": "vectorSearchScore"}}},
        # 3. Rerank: the cross-encoder model reads the query and each document's `embedding_text` together
        #    and reorders the documents by relevance. The model runs on Voyage AI infrastructure, not on the cluster.
        {"$rerank": {
            "query": {"text": query_text},
            "path": "embedding_text",
            "numDocsToRerank": num_docs_to_rerank,
            "model": VOYAGE_RERANK_MODEL,
        }},
        # 4. Read the reranker's relevance score (higher = more relevant to the query).
        #    Docs ($meta "score"): https://www.mongodb.com/docs/manual/reference/operator/aggregation/meta/
        {"$addFields": {"rerank_score": {"$meta": "score"}}},
        # 5. Keep only the best TOP_N documents after reranking.
        {"$limit": top_n},
        # 6. Return readable fields plus both scores; the vector itself is excluded.
        {"$project": {**RESULT_PROJECTION, "vector_score": 1, "rerank_score": 1}},
    ]
    try:
        return list(COLLECTIONS[approach].aggregate(pipeline))
    except OperationFailure as exc:
        # Errors that mention rerank usually mean the feature is not available on this cluster,
        # so add a hint listing the requirements.
        hint = ""
        if "rerank" in str(exc).lower():
            hint = (" Check that the cluster runs MongoDB 9.0+, Native Reranking is enabled in Project Settings,"
                    f" and '{VOYAGE_RERANK_MODEL}' is supported on this cluster tier.")
        raise RuntimeError(f"$rerank pipeline failed: {exc}.{hint}") from exc


# Run the sample queries. If $rerank is unavailable, record that once and skip the rest instead of
# stopping the notebook, so Method 2 and the comparison can still run.
NATIVE_RERANK_AVAILABLE = True
for query in SAMPLE_QUERIES:
    try:
        print_results(
            f"autoEmbed + $rerank | {query}",
            vector_search_with_native_rerank("auto", query),
            score_fields=("vector_score", "rerank_score"),
        )
    except RuntimeError as err:
        NATIVE_RERANK_AVAILABLE = False
        print(f"❌ {err}")
        print("   Skipping the remaining $rerank examples. Method 2 below does not depend on this feature.")
        break

`$rerank` works the same way after the BYOE `$vectorSearch` stage, because it only reads the `embedding_text` field of the incoming documents:

In [ ]:
# Same $rerank stage, but the candidates come from the BYOE collection (search with queryVector).
if NATIVE_RERANK_AVAILABLE:
    print_results(
        f"BYOE + $rerank | {SAMPLE_QUERIES[0]}",
        vector_search_with_native_rerank("byoe", SAMPLE_QUERIES[0]),
        score_fields=("vector_score", "rerank_score"),
    )
else:
    print("⏭️  Skipped: native $rerank is not available on this cluster.")

## 6. Reranking — Method 2: Calling the Rerank API Directly

The same reranker models are available through `POST https://ai.mongodb.com/v1/rerank`. This method fits when:

- the cluster does not meet the `$rerank` requirements;
- candidates come from several sources (for example, vector search plus a cache or another database);
- the application needs to inspect or change candidates before reranking.

The request sends the query and a list of document strings. The response returns `index` (position in the input list) and `relevance_score`, sorted from most to least relevant. `rerank_via_api` maps each `index` back to the original MongoDB document.

In [ ]:
def rerank_via_api(query_text: str, candidates: list[dict[str, Any]], top_k: Optional[int] = None,
                   text_field: str = "embedding_text") -> list[dict[str, Any]]:
    """Rerank candidate documents with the /rerank endpoint and return them with `rerank_score`."""
    # Docs (POST /rerank): https://www.mongodb.com/docs/api/doc/atlas-embedding-and-reranking-api/operation/operation-rerankdocuments
    # Check inputs locally first: the API accepts at most 1000 documents, and empty text cannot be scored.
    if not candidates:
        return []
    if len(candidates) > 1000:
        raise ValueError("The rerank API accepts at most 1000 documents per request")
    missing = [c.get("product_id", "<unknown>") for c in candidates if not c.get(text_field)]
    if missing:
        raise ValueError(f"Candidates missing '{text_field}': {missing}")

    # The API only receives plain strings (one per candidate); MongoDB fields such as _id never leave the app.
    payload = {
        "query": query_text,
        "documents": [c[text_field] for c in candidates],
        "model": VOYAGE_RERANK_MODEL,
        # False: the response contains only index + relevance_score, which keeps it small; the full
        # documents are already available locally in `candidates`.
        "return_documents": False,
        # True: inputs longer than the model's context window are shortened instead of causing an error.
        "truncation": True,
    }
    # top_k asks the API to return only the best N results instead of all of them.
    if top_k is not None:
        payload["top_k"] = top_k

    # Results come back sorted by relevance_score (highest first). `index` is the position of the document
    # in the list that was sent, which maps each result back to its original MongoDB document.
    response = call_voyage_api("rerank", payload)
    reranked = []
    for item in response.get("data", []):
        doc = dict(candidates[item["index"]])
        doc["rerank_score"] = item["relevance_score"]
        reranked.append(doc)
    return reranked


def vector_search_with_api_rerank(approach: str, query_text: str, top_n: int = TOP_N,
                                  num_candidates: int = RERANK_CANDIDATES,
                                  filter_: Optional[dict] = None) -> list[dict[str, Any]]:
    """Retrieve candidates with $vectorSearch, then rerank them in the application via the API."""
    # 1. Retrieve candidates with vector search. Either approach works, because the reranker only needs text.
    candidates = vector_search(approach, query_text, limit=num_candidates,
                               num_candidates=num_candidates * 10, filter_=filter_)
    # Rename the vector score so it is not confused with the rerank score in the output.
    for doc in candidates:
        doc["vector_score"] = doc.pop("score")
    # 2. Rerank in the application with one API call and keep the top_n results.
    return rerank_via_api(query_text, candidates, top_k=top_n)


# BYOE candidates are used here; vector_search_with_api_rerank("auto", ...) works the same way.
for query in SAMPLE_QUERIES:
    print_results(
        f"BYOE + rerank API | {query}",
        vector_search_with_api_rerank("byoe", query),
        score_fields=("vector_score", "rerank_score"),
    )

## 7. Comparison and Cleanup

The table below shows the top results for one query under each strategy. Vector-only ranking reflects overall semantic similarity. The rerankers read the query and product text together, so they tend to promote the products that best match the specific intent (for example, *trail* running over general running or casual walking).

In [ ]:
def compare_rankings(query_text: str, top_n: int = TOP_N) -> None:
    """Print the top product names from each retrieval strategy side by side."""
    # Each strategy is wrapped in a lambda so it only runs when it is called in the loop below.
    strategies = {
        "BYOE vector": lambda: vector_search("byoe", query_text, limit=top_n),
        "autoEmbed vector": lambda: vector_search("auto", query_text, limit=top_n),
        "autoEmbed + $rerank": lambda: vector_search_with_native_rerank("auto", query_text, top_n=top_n),
        "BYOE + rerank API": lambda: vector_search_with_api_rerank("byoe", query_text, top_n=top_n),
    }
    # Skip the native $rerank column if the earlier check showed it is unavailable on this cluster.
    if not NATIVE_RERANK_AVAILABLE:
        strategies.pop("autoEmbed + $rerank")

    # Run every strategy. An error in one strategy appears in its column instead of stopping the comparison.
    columns = {}
    for label, run in strategies.items():
        try:
            columns[label] = [doc["name"] for doc in run()]
        except (RuntimeError, ValueError) as err:
            columns[label] = [f"error: {err}"[:30]]

    # Print a fixed-width table: one column per strategy, one row per rank.
    width = 32
    print()
    print(f"Query: {query_text}")
    print("Rank  " + "".join(f"{label:<{width}}" for label in columns))
    for rank in range(top_n):
        row = "".join(f"{(names[rank] if rank < len(names) else ''):<{width}}" for names in columns.values())
        print(f"{rank + 1:<6}{row}")


for query in SAMPLE_QUERIES:
    compare_rankings(query)

### Cleanup

Set `CLEANUP = True` to drop the demo collections (their search indexes are dropped with them) and close the client.

In [ ]:
# Off by default so that "Run All" never deletes data by accident.
# Docs (dropping a collection): https://www.mongodb.com/docs/manual/reference/method/db.collection.drop/
CLEANUP = False

if CLEANUP:
    try:
        # Dropping a collection also removes its search indexes, so no separate index cleanup is needed.
        for collection in (byoe_collection, auto_collection):
            collection.drop()
            print(f"🗑️  Dropped collection '{collection.name}'")
    except PyMongoError as exc:
        print(f"❌ Cleanup failed: {exc}")
    finally:
        # Always release the client's connection pool, even if a drop failed.
        mongo_client.close()
        print("🔌 MongoDB client closed")
else:
    print("Cleanup skipped. Set CLEANUP = True to drop the demo collections.")

## Summary

| | Bring Your Own Embeddings | Automated Embedding |
|---|---|---|
| Who generates vectors | The application (Embedding API) | Atlas, at index time and on every insert/update |
| Index field type | `vector` (with `numDimensions`, `similarity`) | `autoEmbed` (with `model`, `modality`) |
| Query input | `queryVector` (embedded by the application) | `query.text` (embedded by Atlas) |
| Best when | Full control over models, dimensions, or vectors from other sources is needed | Simplest setup and automatic sync between text and vectors are the priority |

| | `$rerank` stage | Rerank API |
|---|---|---|
| Where it runs | Inside the aggregation pipeline | In application code |
| Requirements | MongoDB 9.0+, Native Reranking enabled | Model API key only |
| Best when | One query should retrieve and rerank in a single round trip | Candidates come from several sources or the cluster does not support `$rerank` |